[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/06_%E5%85%B1%E5%88%86%E6%95%A3%E3%83%BB%E5%A4%89%E5%8B%95%E4%BF%82%E6%95%B0%E3%83%BB%E6%AD%A3%E8%A6%8F%E8%BF%91%E4%BC%BC.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計3級-06. 共分散・変動係数・二項分布の正規近似

3級で抜けやすい計算系トピックをまとめて押さえます。
- 共分散（相関係数の前段）
- 変動係数（ばらつきの比較）
- 二項分布の正規近似

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')

## 1. 共分散

2つの量が「一緒に大きくなる／小さくなる」傾向を表す数。
$$ s_{xy} = \frac{1}{n}\sum (x_i - \bar{x})(y_i - \bar{y}) $$

正なら正の関係、負なら負の関係。ただし**単位に依存する**のが弱点。

In [ ]:
x = df['数学'].values
y = df['英語'].values
cov = np.mean((x - x.mean()) * (y - y.mean()))   # 手計算(÷n)
print('共分散:', round(cov, 2))
print('np.cov(÷n-1):', round(np.cov(x, y, ddof=0)[0,1], 2))

## 2. 共分散 → 相関係数

共分散を各標準偏差で割ると、単位によらない −1〜1 の**相関係数**になります。
$$ r = \frac{s_{xy}}{s_x \, s_y} $$

In [ ]:
r = cov / (x.std() * y.std())
print('相関係数(共分散から):', round(r, 3))
print('pandas .corr() で確認:', round(df['数学'].corr(df['英語']), 3))

## 3. 変動係数（CV）

標準偏差を平均で割ったもの。**単位やスケールが違うデータのばらつきを比べる**のに使う。
$$ CV = \frac{標準偏差}{平均} $$

例：身長(cm)と体重(kg)、どちらが相対的にばらつくか。

In [ ]:
height = np.array([158, 162, 170, 165, 168, 160])
weight = np.array([48, 55, 70, 60, 65, 50])
for name, d in [('身長', height), ('体重', weight)]:
    cv = d.std() / d.mean()
    print(f'{name}: 標準偏差{d.std():.1f}, 平均{d.mean():.1f}, CV={cv:.3f}')
print('→ CVが大きい方が、平均に対して相対的にばらついている')

## 4. 二項分布の正規近似

二項分布 $B(n, p)$ は、$n$ が大きいと正規分布 $N(np,\ np(1-p))$ に近づきます。
コインを多数回投げたときの表の回数で確かめます。

In [ ]:
n, p = 100, 0.5
k = np.arange(0, n + 1)
binom = stats.binom.pmf(k, n, p)
mu, sigma = n * p, np.sqrt(n * p * (1 - p))
xs = np.linspace(0, n, 300)
normal = stats.norm.pdf(xs, mu, sigma)

plt.bar(k, binom, alpha=0.5, label='二項分布 B(100,0.5)')
plt.plot(xs, normal, 'r-', label=f'正規近似 N({mu:.0f},{sigma**2:.0f})')
plt.legend(); plt.title('二項分布の正規近似'); plt.show()
print(f'平均 np={mu}, 標準偏差 √(np(1-p))={sigma:.2f}')

### 近似を使った確率計算
「表が60回以上」の確率を、二項分布そのものと正規近似で比べます。

In [ ]:
exact = 1 - stats.binom.cdf(59, n, p)
# 連続補正（59.5を境にする）を入れた正規近似
approx = 1 - stats.norm.cdf(59.5, mu, sigma)
print(f'二項分布での厳密値: {exact:.4f}')
print(f'正規近似(連続補正): {approx:.4f}  → ほぼ一致')

---
## 🏆 練習問題

**問1.** `df` の `勉強時間` と `数学` の共分散を手計算の式で求め、
それを標準偏差で割って相関係数になることを確かめよう。

**問2.** A店の日販 `[50,52,48,51,49]`（万円）、B店 `[10,14,8,12,16]`（万円）。
変動係数を比べ、どちらが相対的に安定しているか答えよう。

**問3.** サイコロを180回ふるとき「1の目」が出る回数は二項分布 B(180, 1/6)。
平均と標準偏差を求め、正規近似で「35回以上出る確率」を計算しよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


<details><summary>解答例</summary>

```python
a=df['勉強時間'].values; b=df['数学'].values
cov=np.mean((a-a.mean())*(b-b.mean())); print(cov/(a.std()*b.std()))

for d in [np.array([50,52,48,51,49]), np.array([10,14,8,12,16])]:
    print(d.std()/d.mean())   # B店の方がCV大＝不安定

mu, sig = 180/6, np.sqrt(180*(1/6)*(5/6))
print(mu, sig, 1-stats.norm.cdf(34.5, mu, sig))
```
</details>

🎉 これで**3級の出題範囲をほぼ全てカバー**しました。